# Week 1: Exploratory Data Analysis & Leak-Free Chronological Splitting

This notebook explores the **PaySim** mobile money dataset, documents the severe class imbalance, highlights key behavioral patterns across transaction types, critiques the legacy heuristic rule (`isFlaggedFraud`), and verifies the leak-free chronological train/val/test split.

In [ ]:
import pandas as pd
import numpy as np
from src.config import RAW_DATA_FILE, PROCESSED_DATA_DIR, TARGET_COL, TIME_COL
from src.data_loader import load_raw_data, profile_dataset, load_splits

# 1. Load raw data with memory-downcasted data types
df = load_raw_data()
df.head()

## 1. Class Imbalance & Scale Analysis

- **Total Transactions**: 6,362,620
- **Fraud Transactions**: 8,213 (~0.1291%)
- **Temporal Steps**: 1 to 743 hours (31 days)

> **Interview Note**: Because positive fraud cases represent approximately 1 in 800 transactions, **Accuracy** is a dangerous metric. A trivial baseline predicting 0 for all transactions achieves 99.87% accuracy while detecting 0 frauds. We strictly evaluate with **Precision, Recall, and PR-AUC**.

In [ ]:
profile = profile_dataset(df)
print(f"Total transactions : {profile['total_rows']:,}")
print(f"Memory footprint   : {profile['memory_mb']:.1f} MB")
print(f"Total frauds       : {profile['total_frauds']:,} ({profile['fraud_rate_pct']:.4f}%)")
print(f"Time range (step)  : {profile['step_range'][0]} to {profile['step_range'][1]} hours")

## 2. Fraud Distribution by Transaction Type

PaySim features 5 transaction types: `PAYMENT`, `TRANSFER`, `CASH_OUT`, `DEBIT`, and `CASH_IN`.

**Key Empirical Finding**:
Fraud occurs **exclusively** in `TRANSFER` (4,097 cases) and `CASH_OUT` (4,116 cases). Zero fraud cases exist in `PAYMENT`, `CASH_IN`, or `DEBIT`.

In [ ]:
type_df = pd.DataFrame.from_dict(profile['type_breakdown'], orient='index')
type_df

## 3. Failure of the Legacy Rule (`isFlaggedFraud`)

In PaySim, the business rule flags transfers where `amount > 200,000` in a single event.

**Performance Critique**:
- **Flagged Transactions**: 16
- **True Positives**: 16 (Precision: 100%)
- **False Negatives**: 8,197 frauds missed! (Recall: **0.19%**)

This illustrates why static rules fail: fraudsters learn the threshold or use multi-step transfer/cash-out chains, necessitating an ML solution.

In [ ]:
rule = profile['rule_heuristic']
print(f"Transactions Flagged : {rule['flagged_count']}")
print(f"True Positives       : {rule['true_positives']}")
print(f"Precision            : {rule['precision_pct']:.2f}%")
print(f"Recall               : {rule['recall_pct']:.4f}%")

## 4. Leak-Free Chronological Splitting Verification

We enforce strict temporal boundaries:
- **Train Set**: `step <= 550` (steps 1–550)
- **Validation Set**: `551 <= step <= 650`
- **Test Set**: `step > 650` (steps 651–743)

This guarantees that the model is always evaluated on unseen future transactions, simulating true production conditions.

In [ ]:
train_df, val_df, test_df = load_splits()
print(f"Train : {len(train_df):>9,} rows | steps {train_df[TIME_COL].min():>3} to {train_df[TIME_COL].max():<3} | {train_df[TARGET_COL].sum():>5,} frauds ({train_df[TARGET_COL].mean()*100:.3f}%)")
print(f"Val   : {len(val_df):>9,} rows | steps {val_df[TIME_COL].min():>3} to {val_df[TIME_COL].max():<3} | {val_df[TARGET_COL].sum():>5,} frauds ({val_df[TARGET_COL].mean()*100:.3f}%)")
print(f"Test  : {len(test_df):>9,} rows | steps {test_df[TIME_COL].min():>3} to {test_df[TIME_COL].max():<3} | {test_df[TARGET_COL].sum():>5,} frauds ({test_df[TARGET_COL].mean()*100:.3f}%)")

# Assert strict temporal invariants
assert train_df[TIME_COL].max() < val_df[TIME_COL].min()
assert val_df[TIME_COL].max() < test_df[TIME_COL].min()
print("\n✓ Verified: Zero temporal data leakage across splits!")